<a href="https://colab.research.google.com/github/ChrisJavier/UIDE_3-WorkGroupProyectsVisu/blob/main/Week2/src/VADDC_CP_W2_E4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Trabajo 2: # Ejercicio práctico clase 2 (Corregido)

# 📋 Información del Proyecto
- Autores:
  - CARRERA DIAZ CHRISTIAN JAVIER
  - QUISILEMA MEDRANO ALVARO MIJAEL
  - CARDENAS HERRERA EDISON GONZALO
- Versión: 1.0.0
- Licencia: MIT

# 🎯 Introducción:


Han encontrado un conjunto de datos a disposición de la policía de Chicago de 2017 con información sobre los delitos cometidos en toda la ciudad.

Van a leer y ver su conjunto de datos. Este conjunto de datos se descarga de este sitio web. Contiene los incidentes delictivos denunciados (a excepción de los asesinatos, de los que existen datos para cada víctima) que se produjeron en la ciudad de Chicago en 2017.

En base a esa información y los gráficos que realizaremos responderemos la pregunta:

##**¿Cómo desplegar la policía de Chicago para luchar eficazmente contra la delincuencia?**

# 🎯 Sugerencias:

- En el gráfico de la tarea #1 (Cantidad de ocurrencias por tipo de delito - Chicago 2017), se podría colocar la configuración de hovermode='x' para poder ver mas fácil el valor de las categorías de robo que tienen menos incidencias.

- En el gráfico de la tarea #2  se podría hacer barras apiladas para que no se sobrepongan unas barras sobre de otras.

# ⚙️ Recursos útiles para el desarrollo de esta práctica:

Gráficos circulares:
* https://matplotlib.org/stable/gallery/pie_and_polar_charts/pie_features.html
* https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.plot.pie.html

Gráficos en seaborn:
https://python-charts.com/es/seaborn/

Gráficar en ejes matplotlib usando seaborn:
https://stackoverflow.com/questions/23969619/plotting-with-seaborn-using-the-matplotlib-object-oriented-interface

Estructura de pandas dataframe aceptada por seaborn: https://seaborn.pydata.org/tutorial/data_structure.html

## 📊 **Exploración de datos del dataset**

### ⚙️ Importación de librerias

In [ ]:
import pandas as pd
import numpy  as np
import plotly.graph_objects as go
import branca
import geopandas
import folium
import os

import plotly.express as px

from wordcloud import WordCloud #
from IPython.display import display

### ⚙️ Descarga de archivos necesarios para el proyecto

In [ ]:
# Define el nombre del archivo comprimido que se descargará
chicago_crime_Data = 'Chicago_crime_data.csv'
boundaries_beat_Data = 'Boundaries_beat.geojson'
# Descarga el archivo csv

if not os.path.exists(chicago_crime_Data) and not os.path.exists(chicago_crime_Data):
  !wget --timeout=15 --tries=2 'https://raw.githubusercontent.com/ChrisJavier/UIDE_3-WorkGroupProyectsVisu/refs/heads/main/Week2/dataset/Chicago_crime_data.csv' -O '{chicago_crime_Data}'

if not os.path.exists(boundaries_beat_Data) and not os.path.exists(boundaries_beat_Data):
  !wget --timeout=15 --tries=2 'https://raw.githubusercontent.com/ChrisJavier/UIDE_3-WorkGroupProyectsVisu/refs/heads/main/Week2/dataset/Boundaries_beat.geojson' -O '{boundaries_beat_Data}'
# Leer el archivo
df = pd.read_csv(chicago_crime_Data, sep=',', dtype={'ID': object, 'beat_num': object})
df

Podemos observar en la anterior tabla contiene 22 columnas y hay 268.303 registros en total. A continuación se ofrece una breve descripción de cada columna:

|     Variable name    |                    Descripción de la variable                    |                                                                                                  Nota                                                                                                  |
|:--------------------:|:---------------------------------------------------------------:|:------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------:|
|          ID          |              Identificador único para el registro              |                                                                   Cada víctima en un solo caso de homicidio se asigna a un ID diferente                                                                  |
|      Case Number     | El número RD (División de Registros) del Departamento de Policía de Chicago |                                                  Único para el incidente. Múltiples IDs pueden compartir el mismo número de caso si el incidente es un caso de homicidio                                                 |
|         Date         |               Fecha en que ocurrió el incidente              |                                                                                Podría ser una mejor estimación para algunos registros                                                                               |
|         Block        |     La dirección parcialmente redactada donde ocurrió el incidente     |                                                                    La dirección redactada está en el mismo bloque que la dirección real                                                                    |
|         IUCR         |          El código de informe uniforme de crímenes de Illinois          |                                Vinculado directamente al tipo principal y la descripción del delito. Ver detalles [aquí](https://data.cityofchicago.org/widgets/c7ck-438e)                                |
|     Primary Type     |          La descripción principal del código IUCR          |      -                                                                                                                                                                                                  |
|      Description     |         La descripción secundaria del código IUCR         |     -                                                                                                                                                                                                   |
| Location Description |   Descripción de la ubicación donde ocurrió el incidente  |    -                                                                                                                                                                                                    |
|        Arrest        |                  Si se realizó un arresto                 |     -                                                                                                                                                                                                   |
|       Domestic       |          Si el incidente estaba relacionado con violencia doméstica         |                                                                La definición de relacionado con violencia doméstica se basa en la Ley de Violencia Doméstica de Illinois                                                               |
|       beat_num       |         El área de patrulla policial donde ocurrió el incidente        |          El área geográfica policial más pequeña - cada área de patrulla tiene un coche policial dedicado. Ver detalles [aquí](https://data.cityofchicago.org/Public-Safety/Boundaries-Police-Beats-current-/aerh-rz74)         |
|       District       |       El distrito policial donde ocurrió el incidente      | Tres a cinco áreas de patrulla conforman un sector policial y tres sectores conforman un distrito policial. Ver detalles [aquí](https://data.cityofchicago.org/Public-Safety/Boundaries-Police-Districts-current-/fthy-xz3r) |
|         Ward         |            La zona donde ocurrió el incidente            |                          Las zonas son distritos del consejo municipal. Ver detalles [aquí](https://data.cityofchicago.org/Facilities-Geographic-Boundaries/Boundaries-Wards-2015-/sp34-6z76)                          |
|    Community Area    |       El área comunitaria donde ocurrió el incidente       |                                     Ver detalles [aquí](https://data.cityofchicago.org/Facilities-Geographic-Boundaries/Boundaries-Community-Areas-current-/cauq-8yn6)                                    |
|       FBI Code       |   La clasificación del crimen según lo descrito en el NIBRS del FBI  |                         NIBRS significa Sistema Nacional de Informes de Incidentes Basados (NIBRS). Ver detalles [aquí](http://gis.chicagopolice.org/clearmap_crime_sums/crime_types.html)                         |
|       Latitude       |  La latitud de la ubicación donde ocurrió el incidente  |                                                   Esta ubicación se desplaza de la ubicación real para redacción parcial pero cae en el mismo bloque                                                  |
|       Longitude      |  La longitud de la ubicación donde ocurrió el incidente |       -

### ⚙️ Configuración de celda para las visualizaciones

In [ ]:
def enable_plotly_in_cell():
  import IPython
  from plotly.offline import init_notebook_mode
  display(IPython.core.display.HTML('''
        <script src="/static/components/requirejs/require.js"></script>
  '''))
  init_notebook_mode(connected=False) # modo offline

get_ipython().events.register('pre_run_cell', enable_plotly_in_cell)

## 1️⃣ **Fase 1: Investigando crímenes por tipo y descripción.**

Para iniciar nuestro análisis vamos a explorar la relación entre las variables "primary_type" y "description". Como ambas variables son discretas podemos contar el número total de registros que pertenecen a categorías específicas. Tenga en cuenta que cada tipo de "primary_type" tiene su propio conjunto de descripciones que no se solapan. Es decir si dos delitos tienen diferentes "primary_type" entonces no tendrán la misma descripción por definición (a esto se le conoce como variables anidadas).



### 🚀 **Tarea 1**: Crea un gráfico que permita observar la cantidad de ocurrencias por delito. (Se agrega sugerencia)

**Pregunta 1**: Qué pueden identificar??

In [ ]:
# Coloca el código aquí ############
# Tarea 1 - Ocurrencias por delito

import plotly.express as px

# 1: Contar cuántas veces aparece cada tipo de delito
data_t1 = df["Primary Type"].value_counts().reset_index()
# resultado: columnas ["Primary Type", "count"]

# 2: Crear el gráfico de barras interactivo con Plotly
fig = px.bar(
    data_t1,
    x="Primary Type",        # eje X: tipo de delito
    y="count",               # eje Y: número de ocurrencias
    title="Cantidad de ocurrencias por tipo de delito - Chicago 2017",
    color="count",           # colorea las barras según la cantidad
    color_continuous_scale="RdYlGn", #"YlOrRd", #"Reds",  # escala de rojo: más oscuro = más delitos
    width=1000,
    height=700
)

# 3: Mejorar estilo del gráfico
fig.update_layout(
    xaxis_title="Tipo de delito",
    hovermode='x', # Se agrega la sugerencia comentada eliminando el Hover
    yaxis_title="Número de ocurrencias",
    plot_bgcolor="#E0E0E0", # "#3a3a3a",
    xaxis_tickangle=-45,     # inclinar etiquetas para que no se sobrepongan
    coloraxis_showscale=False,  # ocultar barra de color lateral (opcional)
    title_font=dict(size=20, color='darkblue', family='Arial') # Cambia el tamaño, color y tipo de letra del título
)

fig.show()

In [ ]:
# Impresion de conteo por Primary Type para verificar los resultados
print(data_t1.head(10))

#### 🧠 **Interpretación**:

Del gráfico podemos identificar lo siguiente:

1.  Los 3 delitos más frecuentes dominan claramente:

    *   THEFT (~64K): delitos contra la propiedad de baja intensidad — toma de propiedad ajena sin consentimiento del dueño, sin utilizar fuerza, violencia o amenaza contra la persona
    *   BATTERY (~49K): agresiones físicas
    *   CRIMINAL DAMAGE (~29K): daño a la propiedad

2.  Quiebre después del puesto 2:
Existe un salto marcado entre BATTERY (49,218) y CRIMINAL DAMAGE (29,043) — una diferencia de ~20,000 casos. A partir del puesto 3 los valores bajan progresivamente pero sin saltos dramáticos hasta llegar a BURGLARY (13,000), donde se observa un segundo quiebre notable respecto a OTHER OFFENSE (17,235).

3. Cola larga de delitos raros:
Se presenta una cola larga de delitos con frecuencias muy bajas — OBSCENITY (86), NON-CRIMINAL (37), OTHER NARCOTIC VIOLATION (11), PUBLIC INDECENCY (10), HUMAN TRAFFICKING (8) y NON-CRIMINAL (SUBJECT SPECIFIED) (2) tienen menos de 100 casos al año, siendo prácticamente invisibles en el gráfico.

Como sabemos que estas dos variables son anidadas podemos desglosar aún más las frecuencias anteriores incluyendo la descripción.

### 🚀 **Tarea 2**: Escriba código utilizando la función groupby, para contar el número de casos en todas las combinaciones de ``Primary_type`` y ``Description``. A continuación, ordene los resultados en orden decreciente según el número de casos, y grafique en un gráfico de barras agrupado o apilados donde el eje x sea el Primary_type y las barras sean la Description. (Se agrega la sugerencia)

**Pregunta 2**: ¿Cuáles son las descripciones más frecuentes de casos de robo (THEFT), lesiones (Battery) y daños penales (Criminal Damage) en Chicago?

In [ ]:
data_1 = df.groupby(["Primary Type", "Description"])["ID"].count().reset_index(name="count").sort_values(by="count", ascending=False).reset_index(drop=True)
data_1.head()

In [ ]:
len(data_1.head(20)["Description"].unique())

In [ ]:
# Coloquen el código aquí ####################
# Tarea 2 - Descripciones más frecuentes de casos de robo (THEFT), lesiones (Battery) y daños penales (Criminal Damage) en Chicago

# 1. Crear el gráfico
fig = px.bar(
    data_1.head(20),
    x="Primary Type",
    y="count",
    title="Casos más frecuentes por tipo de delito - Chicago 2017",
    color="Description",
    barmode="stack")

# 2. Etiquetar el gráfico
fig.update_layout(
    bargap=.3,
    xaxis_title="Tipo de delito",
    yaxis_title="Número de ocurrencias",
    plot_bgcolor="#E0E0E0",
    title_font=dict(size=20, color='darkblue', family='Arial'))

fig.update_traces(width=0.4)
fig.show()

#### 🧠 **Interpretación**:

Las descripciones más frecuentes relacionados con THEFT, BATTERY y CRIMINAL DAMAGE están:

1. THEFT — Robo

    *   $500 AND UNDER (24,516): es el más frecuente — robos menores cotidianos.

    *   OVER $500 (15,352): robos de mayor valor.
    *   FROM BUILDING (10,662): robos dentro de edificios.
    *   RETAIL THEFT (10,460): robos en tiendas.

2. BATTERY — Lesiones

    *   DOMESTIC BATTERY SIMPLE (23,819): lidera — violencia doméstica es el principal problema.

    *   SIMPLE (16,185): agresiones en calle sin arma

3. CRIMINAL DAMAGE

    *   TO PROPERTY (13,843) y TO VEHICLE (13,555) son casi iguales: daños a propiedades y autos.

Del análisis de frecuencias, sabemos que hay 310 descripciones en total y presentarlas todas en el gráfico no es viable. Sin embargo, podemos utilizar una herramienta de visualización conocida como nube de palabras (word cloud) para resumir las descripciones predominantes dentro de cada tipo primario. Una nube de palabras visualiza las palabras dentro de una colección de textos (en nuestro caso, los textos son todas las descripciones de un tipo primario específico) y el tamaño de cada palabra es proporcional a la frecuencia con que aparece en la base de datos. A continuación, construimos tres nubes de palabras para los 3 tipos de delitos primarios más frecuentes:

In [ ]:
# wordcloud para primary type definido por un rango o posición del crimen
def wordcloud_crime( df, rank, word):
    df_filter = df[df["Primary Type"]==df["Primary Type"].value_counts().index[rank]]
    text = ' '.join(df_filter['Description'])
    wordcloud = WordCloud(max_font_size=50, max_words=100, background_color="white").generate(text)
    # usamos en este caso matplotlib
    #plt.imshow(wordcloud)
    fig = px.imshow(wordcloud, binary_string=True)
    fig.update_layout(title=f'Nube de palabras para {word}')
    fig.show()

In [ ]:
word_i = df["Primary Type"].value_counts().index[0]
wordcloud_crime( df, 0, word_i)

In [ ]:
word_i = df["Primary Type"].value_counts().index[1]
wordcloud_crime( df, 1, word_i)

In [ ]:
word_i = df["Primary Type"].value_counts().index[2]
wordcloud_crime( df, 2, word_i)

**Pregunta 3**: ¿Cuáles son las palabras más comunes para describir estos tipos de casos?

**RTA:**

Cabe recordar que estas conclusiones se basan en la frecuencia de palabras individuales dentro de las descripciones, no en el conteo exacto de cada descripción completa.

Para el delito de **THEFT**, los conceptos más frecuentes giran en torno a
*   Robos en comercios y tiendas (RETAIL),
*   Robos dentro de edificios (BUILDING), y
*   Modalidades de robo personal como el carterismo (POCKET PICKING) y arrebato de bolsos (PURSE SNATCHING).

En el caso de **BATTERY**, predominan claramente dos conceptos:
*   La violencia doméstica (DOMESTIC), y
*   Las agresiones simples sin arma (SIMPLE),
*   También aparecen con relevancia los casos agravados con armas como pistolas (HANDGUN), y
*   Cuchillos (KNIFE), así como víctimas en situación de vulnerabilidad como adultos mayores (SENIOR CITIZEN).

Finalmente, para **CRIMINAL DAMAGE**, los daños más comunes son hacia:
*   Propiedades privadas (PROPERTY) y vehículos (VEHICLE),
*   Seguidos por actos de vandalismo y grafitis (DEFACEMENT),
*   Con una presencia menor de daños a bienes públicos de la ciudad (CITY, CHICAGO, STATE).

Como hemos visto con las nubes de palabras, parece que un determinado tipo de delito suele estar relacionado con ciertos tipos de lugares (por ejemplo, en casa, en tiendas).

### 🚀 **Tarea 3:** Creen una gráfica para investigar los patrones de delincuencia asociados a los tipos de lugares de los delitos (Location Description).

**Pregunta 4:** Basándote en los resultados, ¿qué tipos de lugares son más propensos a la delincuencia?

In [ ]:
# Coloca el código aquí ############
# Tarea 3 — Patrones de delincuencia por lugar

# 1: Contar ocurrencias por lugar
data_t3 = df["Location Description"].value_counts().reset_index()

# 2: Flitrar el top 15 debido a la gran cantidad de lugares
data_t3 = data_t3.head(15)

# 3: Graficar
fig = px.bar(
    data_t3,
    x="count",
    y="Location Description",     # eje Y: lugar del delito
    orientation="h",              # barras horizontales — mejor para textos largos
    title="Top 15 lugares con más delitos - Chicago 2017",
    color="count",
    color_continuous_scale="RdYlGn",
    width=900,
    height=600
)

fig.update_layout(
    xaxis_title="Número de ocurrencias",
    yaxis_title="Lugar del delito",
    yaxis=dict(autorange="reversed"),  # el más frecuente arriba
    coloraxis_showscale=False,
    plot_bgcolor="#E0E0E0",
    font=dict(color="black"),
    title_font=dict(size=20, color='darkblue', family='Arial')
)

fig.show()

#### 🧠 **Interpretación**:

Con respecto a los sitios más propensos a la delincuencia se puede indicar:
1. Se tiene un Top 3 de lugares críticos cuya contabilización de hechos está muy por encima del resto:

*   1. **STREET** con 59,975 ocurrencias — el más peligroso por ser un espacio público abierto donde confluyen todo tipo de delitos
*   2. **RESIDENCE** con 45,880 ocurrencias — delitos en hogares, confirmando el alto índice de violencia doméstica
*   3. **APARTMENT** con 33,448 ocurrencias — delitos en edificios residenciales

2. Un aspecto a considerar con el análisis del gráfico es que tanto APARTMENT como RESIDENCE tienen un alto indice de frecuencia y al ser parte de la categoria **DOMESTIC BATTERY SIMPLE** se puede justifica la razón por la que éste delito es uno de los que tiene mayor ocurrencia.

3. SIDEWALK con 21,006 ocurrencias marca una caída significativa respecto al Top 3, y del puesto 5 en adelante todos los lugares están por debajo de 12,000 casos, siendo OTHER (11,327) y PARKING LOT/GARAGE (8,247) los siguientes más relevantes.

Hasta ahora, hemos visto patrones delictivos vinculados con el "Tipo principal" y la "Descripción de la ubicación" por separado. Tiene sentido ver si una determinada combinación de tipo de delito y tipo de ubicación es frecuente o no. Sabemos que tanto el "Tipo de delito" como la "Descripción del lugar" son variables discretas. Por lo tanto, podemos utilizar una **tabla de contingencia** (tabla cruzada) para resumir el número total de incidentes que pertenecen a una combinación específica de valores de "Primary Type" y "Location Description".

A diferencia del análisis realizado con "Primary Type" y "Description", la `Location Description` y el `Primary Type` **NO** son variables anidadas. Por lo tanto, podemos utilizar la función `crosstab` de `pandas` para generar la tabla de contingencia de dos variables.

### 🚀 **Tarea 4**: La función `crosstab(var1, var2)` genera la tabla de contingencia para `var1` vs. `var2`. Utilice esta función para generar una tabla de contingencia para `Primary Type` vs `Location Description` en la que primero sólo se incluyan las 10 ubicaciones y tipos de delitos más frecuentes.



In [ ]:
# Coloca el código aquí ############
# Tarea 4 — Tabla de contingencia Primary Type vs Location Description

# 1: Filtrar solo los 10 tipos de delito más frecuentes
top10_types = df["Primary Type"].value_counts().head(10).index

# 2: Filtrar solo las 10 ubicaciones más frecuentes
top10_locations = df["Location Description"].value_counts().head(10).index

# 3: Filtrar el dataframe con solo esos top 10
df_filtered = df[
    df["Primary Type"].isin(top10_types) &
    df["Location Description"].isin(top10_locations)
]

# 4: Generar la tabla de contingencia
contingency_table = pd.crosstab(
    df_filtered["Primary Type"],        # filas
    df_filtered["Location Description"] # columnas
)

contingency_table

#### 🧠 **Interpretación**:

La tabla de contingencia muestra que los delitos más frecuentes son theft (robos), battery (agresiones) y criminal damage, concentrándose principalmente en calles (street), residencias (residence) y apartamentos (apartment). Se observa que la violencia ocurre tanto en espacios privados como públicos, mientras que los robos predominan en zonas urbanas con alta actividad como calles y comercios. Además, delitos específicos como el robo de vehículos se concentran en calles y estacionamientos, y los allanamientos en viviendas. En conjunto, el patrón indica que la ocurrencia delictiva depende tanto del tipo de delito como del entorno donde se desarrolla.

### 🚀 **Tarea 5**: Después use una figura de mapa de calor (Heatmap) a través del API graph_objects (go) de Plotly para visualizar la tabla de contingencia.

**Pregunta 5**: Basándose en la tabla de contingencia, ¿cuáles son los puntos calientes de los 10 tipos de delitos más frecuentes? ¿Son los mismos o no?

In [ ]:
# Coloca el código aquí ############
# Tarea 5 — Heatmap con go.Heatmap

import plotly.graph_objects as go

# 1. Generar mapa de calor
fig = go.Figure(data=go.Heatmap(
    z=contingency_table.values,           # valores de la tabla
    x=contingency_table.columns.tolist(), # ubicaciones (eje X)
    y=contingency_table.index.tolist(),   # tipos de delito (eje Y)
    colorscale="Viridis", #RdYlGn
    text=contingency_table.values,        # mostrar números en cada celda
    texttemplate="%{text}",               # formato del texto
))

# 2. Etiquetar
fig.update_layout(
    title="Heatmap — Primary Type vs Location Description (Top 10)",
    xaxis_title="Lugar del delito",
    yaxis_title="Tipo de delito",
    xaxis_tickangle=-45,
    width=1100,                           # Tamaño Grande
    height=700,
    paper_bgcolor="#2d2d2d",
    plot_bgcolor="#3a3a3a",
    font=dict(color="white"),
)

fig.show()


#### 🧠 **Interpretación**:

Los puntos calientes varían según el tipo de delito.

| Delito | Punto caliente principal | Conteo |
|:---|:---|---:|
| **THEFT** | `STREET` | 15,801 |
| **BATTERY** | `APARTMENT` | 11,706 |
| **CRIMINAL DAMAGE** | `STREET` | 9,997 |
| **MOTOR VEHICLE THEFT** | `STREET` | 8,217 |
| **DECEPTIVE PRACTICE** | `RESIDENCE` | 6,225 |
| **OTHER OFFENSE** | `RESIDENCE` | 6,697 |
| **BURGLARY** | `RESIDENCE` | 4,170 |
| **ASSAULT** | `STREET` | 3,533 |
| **ROBBERY** | `STREET` | 3,513 |
| **NARCOTICS** | `SIDEWALK` | 3,264 |

Esto indica que no existe un único punto caliente pero si existen puntos en común entre varios tipos de delito. Con el análisis realizado se puede ver de inmediato que:

*   6 de 10 delitos tienen su punto caliente en STREET o espacios públicos (SIDEWALK).
*   3 de 10 delitos lo tienen en entornos privados que pueden ser RESIDENCE y APARTMENT.
*   1 de 10 delitos tiene un patrón único en SIDEWALK.

**Pregunta 6:** Con la información que ha descubierto hasta ahora como puede desplegar los policias de Chicago.



**RTA**:

Con base en el análisis realizado en la Fase 1, se pueden identificar tres estrategias claras de despliegue policial.

En primer lugar, dado que **THEFT** es el delito más frecuente con 64,356 casos y se concentra principalmente en STREET (59,975 ocurrencias en total para todos los delitos), se recomienda intensificar el patrullaje en vías públicas y zonas comerciales, especialmente para prevenir robos menores ($500 AND UNDER con 24,516 casos), robos en edificios (FROM BUILDING con 10,662 casos) y robos en tiendas (RETAIL THEFT con 10,460 casos).

En segundo lugar, **BATTERY** representa el segundo delito más frecuente con 49,218 casos y su punto caliente es APARTMENT, donde DOMESTIC BATTERY SIMPLE lidera con 23,819 casos, lo que justifica la necesidad de desplegar unidades especializadas en violencia doméstica con capacidad de respuesta en zonas residenciales.

En tercer lugar, **CRIMINAL DAMAGE** con 29,043 casos se concentra mayoritariamente en STREET y se manifiesta principalmente como daños a propiedades (TO PROPERTY con 13,843 casos) y vehículos (TO VEHICLE con 13,555 casos), por lo que el patrullaje en vías públicas no solo reduciría los robos sino que simultáneamente actuaría como factor disuasivo para el vandalismo, maximizando el impacto de cada unidad desplegada.

En conclusión, una estrategia de patrullaje intensivo en calles y zonas residenciales cubriría los tres delitos más frecuentes de Chicago, optimizando el uso de los recursos policiales disponibles.

## 2️⃣ Fase 2: Investigando crímenes a través del tiempo.

Pasamos ahora a investigar la relación entre los incidentes delictivos y el tiempo; es decir, la variable `Date`. El tiempo es una de las dimensiones más importantes para construir un plan de despliegue eficaz. Dado que no podemos patrullar todos los lugares las 24 horas del día, los 7 días de la semana, debemos centrarnos en los periodos de tiempo con altos índices de delincuencia. La "fecha" nos proporciona una marca de tiempo para cada incidente, lo que nos permite contar cuántos incidentes se produjeron en un periodo de tiempo determinado. Como disponemos de los datos de un año, podemos empezar con el total mensual de incidentes para ver si ciertos meses son propensos a la delincuencia.

Vamos a cubrir diferentes temporalidades Día, Semana y Mes para tener varías perspectivas. Tenga en cuenta que en Chicago el clima juega un papel importante.

In [ ]:
# Primero convertir la columna Date a formato fecha
df["date_py"] = pd.to_datetime(df.Date, format="mixed")

In [ ]:
# EDA básico
print(df.shape)
print(df.info())
print(df.isnull().sum())
print(df.duplicated().sum())

### 🚀 **Tarea 6:** Gráfica el número de delitos en general a través del tiempo, agrupando por mes. Puedes usar pandas para agrupar por mes y contar el número de delitos.

**Pregunta 7:** ¿Qué meses tienen tasas de delincuencia relativamente más altas?

In [ ]:
# Coloca el código aquí ############
# Tarea 6: Crímenes por mes

# Agrupamos los incidentes por mes y contamos el total de ocurrencias
por_mes = df.groupby(df["date_py"].dt.month).size().reset_index(name="Count")
por_mes.columns = ["Mes", "Count"]  # Renombramos las columnas para mayor claridad

# Creamos un gráfico de línea con Plotly para visualizar la tendencia mensual
fig = px.line(por_mes, x="Mes", y="Count", title="Total de incidentes mensuales")

# Reemplazamos los números de mes por nombres abreviados en el eje X
fig.update_xaxes(tickvals=list(range(1,13)),
                 ticktext=["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"])
fig.show()

#### 🧠 **Interpretación**:

El gráfico de línea muestra el total acumulado de incidentes agrupados por mes a lo largo de todo el dataset. Se observa que los meses de verano, especialmente julio (24,817) y agosto (24,683), concentran la mayor actividad delictiva, mientras que febrero registra el mínimo con 19,270 incidentes. Este patrón sugiere una tendencia estacional donde los meses cálidos presentan mayores tasas de delincuencia, lo cual es consistente con el clima extremo de Chicago donde los inviernos fríos reducen la actividad en las calles. Sin embargo, es importante considerar que este conteo no está normalizado por año ni por número de días del mes, por lo que estas diferencias deben interpretarse como tendencias generales.

### 🚀 **Tarea 7:** Gráfica el número de delitos en general a través del tiempo, agrupando por día. Puedes usar pandas para agrupar por día y contar el número de delitos.

**Pregunta 8:** ¿Sigue creyendo que febrero es la época en que la delincuencia es menos preocupante?

In [ ]:
# Coloca el código aquí ############
# Tarea 7: Crímenes por día

# Agrupamos los incidentes por día calendario y contamos el total de ocurrencias
por_dia = df.groupby(df["date_py"].dt.date).size().reset_index(name="Count")
por_dia.columns = ["Dia", "Count"]  # Renombramos las columnas para mayor claridad

# Creamos un gráfico de línea para visualizar la variación diaria a lo largo del año
fig = px.line(por_dia, x="Dia", y="Count", title="Total de incidentes por día")
fig.show()


#### 🧠 **Interpretación**:

El gráfico de incidentes por día muestra la variación diaria a lo largo de todo el dataset, permitiendo observar picos y caídas puntuales que no son visibles en la agrupación mensual. Si bien el gráfico mensual anterior sugería que febrero tiene el menor número total de incidentes, al ver la tendencia diaria se puede apreciar que esta cifra está influenciada por el hecho de que febrero es el mes más corto del año con solo 28 días. Por lo tanto, no sería correcto afirmar que febrero es necesariamente la época menos preocupante, ya que su menor conteo total puede deberse simplemente a tener menos días y no a una reducción real en la tasa diaria de delincuencia.
Para determinar realmente si tiene este efecto, vamos a calcular y graficar el promedio diario de crímenes para poder analizar correctamente la tendencia.

In [ ]:
# Promedio diario de crímenes por mes
dias_por_mes = df.groupby(df["date_py"].dt.month)["date_py"].apply(lambda x: x.dt.date.nunique())
total_por_mes = df.groupby(df["date_py"].dt.month).size()

promedio_diario = (total_por_mes / dias_por_mes).reset_index(name="Promedio")
promedio_diario.columns = ["Mes", "Promedio"]

fig = px.line(promedio_diario, x="Mes", y="Promedio", title="Promedio diario de incidentes por mes")
fig.update_xaxes(tickvals=list(range(1,13)),
                 ticktext=["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"])
fig.show()

Al calcular el promedio diario de crímenes por mes, se confirma que febrero y marzo presentan los promedios diarios más bajos del año. Esto indica que la baja delincuencia en estos meses no se debe a que febrero tenga menos días, sino que refleja un efecto real del clima: los inviernos extremadamente fríos de Chicago reducen la actividad en las calles y con ello los incidentes delictivos. Por lo tanto, se puede afirmar con mayor confianza que los meses de invierno, especialmente febrero y marzo, son efectivamente los períodos de menor preocupación en términos de delincuencia.

### 🚀 **Tarea 8:** Gráfica el número de delitos en general a través del tiempo, agrupando por día de la semana. Puedes usar pandas para agrupar por día de la semana y contar el número de delitos.

**Pregunta 9:** ¿Cuáles son los días con mas y menos delitos?


In [ ]:
# Coloca el código aquí ############
# Tarea 8: Crímenes por día de la semana

# Agrupamos los incidentes por día de la semana y contamos el total de ocurrencias
por_semana = df.groupby(df["date_py"].dt.dayofweek).size().reset_index(name="Count")
por_semana.columns = ["Dia", "Count"]

# Reemplazamos el número de día por su nombre (0=Lunes, 6=Domingo)
por_semana["Dia"] = ["Lunes","Martes","Miércoles","Jueves","Viernes","Sábado","Domingo"]

# Gráfico de línea con marcadores para visualizar mejor las diferencias entre días
fig = px.line(por_semana, x="Dia", y="Count", title="Total de incidentes por día de la semana",
              markers=True)
fig.show()

#### 🧠 **Interpretación**:

El gráfico de línea por día de la semana revela un patrón muy claro: el viernes es el día con mayor número de incidentes delictivos con 40,132 casos, destacándose notablemente del resto con un pico pronunciado. En el extremo opuesto, el miércoles registra la menor cantidad con 37,565 incidentes. Se observa además una tendencia interesante: los días de mitad de semana (martes, miércoles y jueves) se mantienen relativamente bajos y estables, mientras que el viernes dispara significativamente, posiblemente asociado al inicio del fin de semana, mayor actividad nocturna y consumo de alcohol. El sábado y domingo bajan respecto al viernes pero se mantienen por encima de los días entre semana.

## 3️⃣ Fase 3: Investigando crímenes por posición geográfica.

Otra dimensión importante que debemos considerar es la relación entre los incidentes delictivos y la ubicación geográfica. El aspecto técnico de este análisis puede verse como una profundización de lo que hemos aprendido previamente sobre visualizaciones de mapas.

Disponemos de las coordenadas geográficas aproximadas de cada incidente y, a partir de ellas, podemos explorar los patrones geográficos de la delincuencia en Chicago. Para identificar los focos geográficos de delincuencia, podemos dividir la ciudad de Chicago en regiones no superpuestas y contar el número total de casos en 2017 en cada región.En este caso, dividimos Chicago por distritos policiales y se cuentan los delitos por rondas policiales . A continuación, visualizamos los resultados en los mapas usando las librerías Plotly, Folium y geopandas:

**Pregunta 10:** Cuales puntos calientes identificaron?

Se usaron librerías adicionales ya que plotly presenta algunas complejidades y limitaciones para la visualización de mapas.

1. Con plotly se gráfica la distribución de puntos geo.

2. Se usa la librería geopandas para cargar los polígonos de las rondas policiales y cruzarlos con los puntos lon y lat y determinar asi el número de crimenes por polígono.

3. Seguidamente folium se utiliza para graficar esta información sobre un mapa de open street map, el cual es de acceso gratuito.

In [ ]:
# graficar una muestra de la distribución de puntos sobre el mapa
# de open street usando plotly para el tipo de delito mas comun
filter_df = df.loc[df['Primary Type'].isin(['THEFT', 'CRIMINAL DAMAGE'])]
# Eliminamos los datos que esten NaN para mayor precision
filter_df = filter_df.dropna(subset=['Latitude', 'Longitude'])

#'BATTERY', 'ASSAULT', 'CRIMINAL DAMAGE']
fig = px.density_mapbox(filter_df,
                        lat='Latitude',
                        lon='Longitude',
                        z=None,
                        radius=1,
                        zoom=9.5,
                        mapbox_style="open-street-map")

# Configurar el diseño del mapa
fig.update_layout(title='Distribución geo de delitos')

Para una funcionalidad más profunda involucrando un análisis espacial menos saturado y poder responder la pregunta 10, se usa las librerías geopandas y folium.

In [ ]:
#formatear la variable rondas policiales
df["beat_num"] = df["beat_num"].str.zfill(4)
# por rondas contar el numero de crimenes
beat_cn = df.groupby("beat_num")["ID"].count().reset_index(name="crime_count")
# Definir el esquema de color
min_cn, max_cn = beat_cn['crime_count'].quantile([0.01,0.99]).apply(round, 2)
# Crear la paleta de colores
colormap = branca.colormap.LinearColormap(
    colors=['white','yellow','orange','red','darkred'],
    #index=beat_cn['count'].quantile([0.2,0.4,0.6,0.8]),b
    vmin=min_cn,
    vmax=max_cn
)
colormap.caption="Total crimenes en Chicago por rondas policiales"

# cargar los poligonos de los sectores y limites en donde se hacen las rondas policiales
beat_orig = geopandas.read_file(boundaries_beat_Data, driver = "GeoJSON")
# se interseca con los puntos
beat_data = beat_orig.join(beat_cn.set_index("beat_num"), how = "left", on = "beat_num")
beat_data.fillna(0, inplace = True)

In [ ]:
# Visualización interactiva del crimen por rondas policiales usando folium
m_crime = folium.Map(location=[41.88, -87.63],
                        zoom_start=12,
                        tiles="OpenStreetMap")
style_function = lambda x: {
    'fillColor': colormap(x['properties']['crime_count']),
    'color': 'black',
    'weight':2,
    'fillOpacity':0.5
}

stategeo = folium.GeoJson(
    beat_data.to_json(),
    name='Rondas policiales',
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=['beat_num', 'crime_count'],
        aliases=['Ronda', 'Crimen total'],
        localize=True
    )
).add_to(m_crime)

colormap.add_to(m_crime)
m_crime

#### 🧠 **Interpretación**:

Según el primer gráfico tenemos una densidad en los delitos com mayor concentracion en zonas urbanas centrales, en especial al sector este de la cidad. En el patrón del gráfico de densidad poblacional, actividad urbana y ocurrencia delictiva pero por otro lado se muestra que tiene menor insidencia de delincuencia.

Por otro lado, en el mapa coroplètico tiene una forma de interpretarlo màs estructural donde el color muestra el total de crimenes en el area seleccionada. Las zonas centro-sur de la ciudad muestra mayor concentracion, por el mapa de densidad anterior se muestra la misma informacion donde la diferencia es separada màs cuadrificada. El patròn sugiere una relacion entre la actividad humana, la densidad poblacional y la incidencia delictiva.


Ahora poniendo un ejemplo con una normalizacion de la data para verlo asi en el mapa tenemos:


In [ ]:
# Importamos la libreria KMeans para hacer un cluster espacial
from sklearn.cluster import KMeans

# Obtenemos los datos de las coordenadas eliminando los valores NaN
df_normalized = df.dropna(subset=['Latitude', 'Longitude'])

# Limitamos el dataset para evidar desbordamiento en Google colab
df_normalized = df_normalized.sample(n=10000, random_state=42)

# Realizamos el gluster de 5 con la semilla de 5
kmeans = KMeans(n_clusters=5, random_state=42)

df_normalized['cluster'] = kmeans.fit_predict(df_normalized[['Latitude', 'Longitude']])

fig = px.scatter_mapbox(
    df_normalized,
    lat="Latitude",
    lon="Longitude",
    color="cluster",
    zoom=10,
    mapbox_style="open-street-map",
    title="Zonas de riesgo (clustering geoespacial)"
)

fig.show()

Con la normalizacion realizada gracias al clustering geoespacial nos revela que la distribución del crimen presenta una estructura espacial no aleatoria, con agrupaciones claramente definidas que corresponden a diferentes zonas urbanas. Estas agrupaciones pueden interpretarse como regiones con dinámicas delictivas similares, lo que permite segmentar la ciudad en áreas de riesgo diferenciadas sin depender de divisiones administrativas predefinidas.

El gráfico muestra que el crimen se agrupa en regiones geográficas definidas, evidenciando patrones espaciales claros que permiten segmentar la ciudad en zonas con dinámicas delictivas similares.

**Pregunta 11**: Basándose en el análisis realizado hasta ahora, ¿cuál es su estrategia para el despliegue policial en función del tiempo, los lugares y los tipos de delitos?


#### 🧠 **Interpretación**:

Como parte de nuestro grupo concluimos que la mejor estrategia de despliegue policial debe centrarse en una asignación inteligente de recursos basada en patrones espaciales, priorizando las zonas identificadas con la mayor densidad. Estas áreas, principalmente ubicadas en sectores centrales y con alta densidad urbana, concentran la mayor cantidad de incidentes, por lo que requieren patrullaje intensivo y presencia constante. En contraste, las zonas periféricas, con menor densidad delictiva, pueden ser cubiertas con patrullaje intermitente. Además, el análisis por tipo de delito permite focalizar acciones específicas: mayor vigilancia en calles y zonas comerciales para prevenir robos, y presencia en áreas residenciales para mitigar delitos violentos como agresiones.

Como plan de acción, se recomienda implementar un modelo de patrullaje dinámico que combine vigilancia focalizada en los clusters de mayor riesgo con ajustes operativos según franjas horarias críticas. Esto implica incrementar la presencia policial en horarios nocturnos en zonas residenciales para reducir delitos violentos, y reforzar la vigilancia diurna en áreas comerciales para prevenir robos. Adicionalmente, se sugiere complementar el despliegue con estrategias preventivas como monitoreo con cámaras en puntos críticos, coordinación con la comunidad en zonas residenciales y análisis continuo de datos para actualizar los patrones delictivos. Este enfoque permitirá a la policía optimizar recursos, anticiparse a los incidentes y mejorar la efectividad en la seguridad ciudadana.

In [ ]:
# Agregamos el tamaño de crimen total
cluster_counts = df_normalized.groupby('cluster').size().reset_index(name='crime_total')

# Unimos para visualización
df_normalized = df_normalized.merge(cluster_counts, on='cluster')

# Mapa (tamaño = prioridad)
fig = px.scatter_mapbox(
    df_normalized,
    lat="Latitude",
    lon="Longitude",
    color="cluster",
    size="crime_total",
    size_max=15,
    zoom=10,
    mapbox_style="open-street-map",
    title="Sectores prioritarios para despliegue policial"
)


fig.show()